# Build Output File for CIL Tracker

*Key Scopes*
- ADM1 Level
- ISO(GID_0) Level
- Global

Combine *Running Total* and *Forecast* projections to summarize average total deaths relative to a 1996-2025 baseline period.

In [10]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.home() / "projects" / "poreallas"
sys.path.insert(0, str(PROJECT_ROOT))

import importlib
import os

import pandas as pd
import numpy as np
import xarray as xr
from dotenv import find_dotenv, load_dotenv
from datetime import datetime, UTC

from utils import analysis_utils, isku_utils, plotting_utils

importlib.reload(analysis_utils)
importlib.reload(isku_utils)

/home/emily_zuetell/projects/poreallas/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<module 'utils.isku_utils' from '/home/emily_zuetell/projects/poreallas/utils/isku_utils.py'>

In [29]:
load_dotenv(
    find_dotenv("october_local.env", raise_error_if_not_found=True), override=True
)

### Set this to the current projection Month (e.g. 10: October forecast)
PROJ_MONTH = 10

# Define Forecast
EFFECTS_URI = os.environ["POREALLAS_EFFECTS_URI"]
EFFECTS_RT_URI = os.environ["POREALLAS_EFFECTS_RT_URI"]

# Load and Clean ADM1 Regions
ADM1_BASE = pd.read_parquet("../data/base/adm1.parquet")
# Fill Blanks where GID_1 = GID_0
g1 = pd.Series(ADM1_BASE["GID_1"].values, dtype=object)
blank = g1.isna() | (g1.astype(str).str.strip() == "")
ADM1_BASE["GID_1"] = g1.where(~blank, ADM1_BASE["GID_0"].values).to_numpy()

In [18]:
# Build MONTH arrays (w/ wraparound)
MONTHS_RT = [(5 + i) % 12 + 1 for i in range((PROJ_MONTH - 6) % 12)] # Running Total Months (June:MONTH_PROJ-1)
MONTHS_FC = [(PROJ_MONTH - 1 + i) % 12 + 1 for i in range(6)] # Forecast Months (MONTH_PROJ:6)
# Combined Months in Tracker
MONTHS_ALL = np.concat([MONTHS_RT, MONTHS_FC])

# Baseline period (Verify: 1996-2025)
BASELINE_PERIOD = slice("1996-01-01", "2025-12-31")

RATE = False #[False: Total Deaths, True: Mortality Rate] -> Use False for Tracker

print('OUTPUT PARAMETERS')
print("Running Total Months:", MONTHS_RT)
print("Forecast Months:", MONTHS_FC)
print(f"Baseline Period: {BASELINE_PERIOD.start} to {BASELINE_PERIOD.stop}")
print(f"Output Units: {'Mortality Rate' if RATE else "Total Deaths"}")

OUTPUT PARAMETERS
Running Total Months: [6, 7, 8, 9]
Forecast Months: [10, 11, 12, 1, 2, 3]
Baseline Period: 1996-01-01 to 2025-12-31
Output Units: Total Deaths


In [19]:
# Set Configuration [DON"T CHANGE for CIL TRACKER OUTPUT]
# Forecast Configuration #
config_fc = analysis_utils.ImpactConfig(
    version=f"v{datetime.now(tz=UTC):%y%m%d}", # Set version to today
    baseline_period=BASELINE_PERIOD,
    polygons_path=os.environ["POREALLAS_REGIONS_POLYGONS_URI"],
    socioeconomics_path=os.environ["POREALLAS_SOCIOECONOMICS_URI"],
    rate=RATE,
    months=MONTHS_FC,# Forecast Months (6 months)
    hotonly="hotonly", # We are using only hot-only for the tracker
    dims=["number", "sample"],
)
# Running Total Configuation #
config_RT = analysis_utils.ImpactConfig(
    version=f"v{datetime.now(tz=UTC):%y%m%d}_RT",
    baseline_period=BASELINE_PERIOD,
    polygons_path=os.environ["POREALLAS_REGIONS_POLYGONS_URI"],
    socioeconomics_path=os.environ["POREALLAS_SOCIOECONOMICS_URI"],
    rate=RATE,
    months=MONTHS_RT, # Running Total Months (June-)
    hotonly="hotonly",
    dims=["sample"], # No ensemble members in Running Total Data
)

# Combined Running Total + Forecast Configuration
config_all = analysis_utils.ImpactConfig(
    version=f"v{datetime.now(tz=UTC):%y%m%d}",
    baseline_period=BASELINE_PERIOD,
    polygons_path=os.environ["POREALLAS_REGIONS_POLYGONS_URI"],
    socioeconomics_path=os.environ["POREALLAS_SOCIOECONOMICS_URI"],
    rate=False,
    months=MONTHS_ALL,
    hotonly="hotonly",
    dims=[], # Dims are already reduced
)

In [25]:
### Disputed Region Remaps ###
# ISO Z0x mapped to associated country (e.g. IND, CHN)
REMAPS = {
    "BASE": {
        "Z01": "IND",
        "Z02": "CHN",
        "Z03": "CHN",
        "Z04": "IND",
        "Z05": "IND",
        "Z06": "PAK",
        "Z07": "IND",
        "Z08": "CHN",
        "Z09": "IND",
    },
    "IN": {
        "Z01": "IND",
        "Z02": "IND",
        "Z03": "IND",
        "Z04": "IND",
        "Z05": "IND",
        "Z06": "IND",
        "Z07": "IND",
        "Z08": "IND",
        "Z09": "IND",
    },
    "CN": {
        "TWN": "CHN",
        "Z02": "CHN",
        "Z03": "CHN",
        "Z04": "CHN",
        "Z05": "CHN",
        "Z07": "CHN",
        "Z08": "CHN",
        "Z09": "CHN",
        "Z01": "IND",
        "Z06": "PAK",
    },
}

In [20]:
# Open Projection Effects
effect = xr.open_datatree(EFFECTS_URI, consolidated=False)
# Open and Align Running Total
effect_rt = xr.open_datatree(EFFECTS_RT_URI, consolidated=False)
# Insert baseline from 'effect' since baseline does not need to be computed in running total workflow
effect_rt['/baseline'] = effect['/baseline']
effect_rt['/baseline_hotonly'] = effect['/baseline_hotonly']
# Rename forecast for analysis functions
effect_rt['/forecast'] = effect_rt['/forecast_runningtotal']
effect_rt['/forecast_hotonly'] = effect_rt['/forecast_hotonly_runningtotal']

In [21]:
# Compute impact: forecast - baseline
impact = analysis_utils.compute_impact(
    effect.chunk({dim: -1 for dim in config_fc.dims}),
    config_fc,
    ensemble=True,
)
# Use only the defined 6-months
impact = impact.sel(month=config_fc.months)
# Use only mean values
impact = impact.mean(dim = config_fc.dims)

# Compute impact: forecast - baseline
impact_rt = analysis_utils.compute_impact(
    effect_rt.chunk({dim: -1 for dim in config_RT.dims}),
    config_RT,
    ensemble=True,
)
# Use only the defined 6-months
impact_rt = impact_rt.sel(month=config_RT.months)
# Use only mean values
impact_rt = impact_rt.mean(dim = config_RT.dims)

# Concat running total and forecast (We can do this because they are already the mean of sample, number and only have dim region, month)
impact_all = xr.concat([impact_rt, impact], dim="month")

In [22]:
# Aggregte to ADM1 [This needs CIL_regionalization package]
impact_adm1, merge_key, base_cols = analysis_utils.aggregate_impact(impact_all, config_all, 'ADM1')

/home/emily_zuetell/projects/poreallas/.venv/lib/python3.14/site-packages/dask/_task_spec.py:768: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
source coverage: 6 unmatched source units (first 6: [['BRA.8.823.1961'], ['IDN.14.203'], ['IND.12.129.431'], ['IND.12.129.432'], ['VNM.4.37.365.6048'], ['ZAF.9.313']]); recorded in the manifest and skipped
source coverage: 1 absent_from_data source units (first 1: [['ATA']]); recorded in the manifest and skipped


In [23]:
# Fill GID_1 where GID_1 = GID_0
gid1 = pd.Series(impact_adm1["GID_1"].values, dtype=object)
gid0 = pd.Series(impact_adm1["GID_0"].values, dtype=object)
filled = gid1.where(gid1.notna() & (gid1 != ""), gid0)

impact_adm1 = impact_adm1.assign_coords(GID_1=filled.to_numpy())

# Add ADM1 Names
names = ADM1_BASE.drop_duplicates("GID_1").set_index("GID_1").reindex(impact_adm1["GID_1"].values)

impact_adm1 = impact_adm1.assign_coords(
    NAME_0=("GID_1", names["NAME_0"].to_numpy(dtype=object)),
    NAME_1=("GID_1", names["NAME_1"].to_numpy(dtype=object)),
)
# Fill NAME_1 where GID_1 = GID_0
n1 = pd.Series(impact_adm1["NAME_1"].values, dtype=object)
n0 = pd.Series(impact_adm1["NAME_0"].values, dtype=object)
impact_adm1 = impact_adm1.assign_coords(
    NAME_1=("GID_1", n1.where(n1.notna() & (n1 != ""), n0).to_numpy())
)

In [26]:
COLS = ['GID_0', 'GID_1', 'NAME_0', 'NAME_1', 'Time', 'Year', 'Additional Deaths']

def to_frame(x, time=None, year=None, iso=False, is_global=False, round = True):
    # Xarray to Dataframe
    _df = analysis_utils.dataset_to_dataframe(x)
    # Round to 10s, 100s, etc.
    if round:
        _df = analysis_utils.round_output(_df)
    _df = _df.rename(columns={'value': "Additional Deaths"})
    if time is None:  # Monthly rows
        m = _df["month"].astype(int)
        _df["Year"] = m.map(lambda v: 2026 if v >= 6 else 2027)  # Assign Year based on month (no years in projection file yet; fix once it wraps around)
        _df["Time"] = pd.to_datetime(dict(year=_df["Year"], month=m, day=1)).dt.strftime("%B")  # Month Name
    else:  # Cumulative Rows
        _df["Year"] = year
        _df["Time"] = time
    if is_global:
        # No country or region dimension, so fill every geography column
        _df["GID_0"] = "Global"
        _df["NAME_0"] = "Global"
        _df["GID_1"] = "-"
        _df["NAME_1"] = "-"
    elif iso:
        # Enforce blank columns for ISO
        _df["GID_1"] = "-"
        _df["NAME_1"] = "-"
    return _df[COLS]

def add_iso_names(x):
    return x.assign_coords(
        NAME_0=("GID_0", iso_names.reindex(x["GID_0"].values).to_numpy(dtype=object))
    )



In [28]:
TIME_ORDER = [pd.Timestamp(2000, m, 1).strftime("%B") for m in MONTHS_ALL] + [
    "Cumulative", "6 Month Forecast", "Total"
]

ROUND = True

for disp_ver, remap in REMAPS.items():
    # Remap ISO based on version
    gid0 = impact_adm1["GID_0"].to_series().replace(remap)
    da = impact_adm1.assign_coords(GID_0=("GID_1", gid0.to_numpy(dtype=object)))
    da = da.assign_coords({
        k: (v.dims, np.asarray(v.values, dtype=object))
        for k, v in da.coords.items()
        if v.dtype == object or "string" in str(type(v.variable._data))
    })
    iso_names = (
        ADM1_BASE.assign(GID_0=ADM1_BASE["GID_0"].replace(remap))
        .drop_duplicates("GID_0")
        .set_index("GID_0")["NAME_0"]
    )

    # Group to ISO once; NAME_0 carries through
    iso = add_iso_names(da.groupby("GID_0").sum())

    # Worldwide Totals
    global_sum = iso.sum(dim = 'GID_0')

    # Build df_out for ADM1, ISO, Global
    df_out = pd.concat([
        to_frame(da, round = ROUND), # Monthly Mean All Months
        to_frame(da.sel(month=MONTHS_RT).sum("month"), "Cumulative", "2026", round = ROUND), # Cumulative Running Total
        to_frame(da.sel(month=MONTHS_FC).sum("month"), "6 Month Forecast", "2026", round = ROUND), # 6-month Forecast
        to_frame(da.sum("month"), "Total", "2027", round = ROUND), # Total (Running Total + Forecast)
        to_frame(iso, iso=True, round = ROUND), # Monthly Mean All Months
        to_frame(iso.sel(month=MONTHS_RT).sum("month"), "Cumulative", "2026", iso=True, round = ROUND), # Cumulative Running Total
        to_frame(iso.sel(month=MONTHS_FC).sum("month"), "6 Month Forecast", "2026", iso=True, round = ROUND), # 6-month Forecast
        to_frame(iso.sum("month"), "Total", "2027", iso=True, round = ROUND), # Total (Running Total + Forecast)
        to_frame(global_sum, is_global=True, round = ROUND), # Monthly Mean All Months
        to_frame(global_sum.sel(month=MONTHS_RT).sum("month"), "Cumulative", "2026", is_global=True, round = ROUND), # Cumulative Running Total
        to_frame(global_sum.sel(month=MONTHS_FC).sum("month"), "6 Month Forecast", "2026", is_global=True, round = ROUND), # 6-month Forecast
        to_frame(global_sum.sum("month"), "Total", "2027", is_global=True, round = ROUND), # Total (Running Total + Forecast)
    ], ignore_index=True)
    #Build "name" column: NAME_1, NAME_0
    df_out["name"] = (df_out["NAME_1"] + ", " + df_out["NAME_0"]).where(df_out["NAME_1"] != "-", df_out["NAME_0"])
    # Set Column Order
    df_out = df_out[['GID_0', 'GID_1', 'NAME_0', 'NAME_1', 'name', 'Time', 'Year', 'Additional Deaths']]
    # Sort Rows (Global rows first, ADM1 nested below associated ISO)
    df_out["Time"] = pd.Categorical(df_out["Time"], categories=TIME_ORDER, ordered=True)
    df_out["_not_global"] = df_out["GID_0"].ne("Global")
    df_out = (
        df_out.sort_values(["_not_global", "GID_0", "GID_1", "Time"], ignore_index=True)
              .drop(columns="_not_global")
    )

    df_out.to_csv(f"{config_all.version}_adm1_output_{disp_ver}_{'round' if ROUND else 'raw'}.csv", index=False)